## Setup

### Imports

In [74]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
import alphapepttools as at
from scipy.sparse import csr_matrix
from itertools import combinations
from scipy import sparse

In [75]:
prec_report_path = "../data/diann_1.8.1_report_head.tsv"

In [ ]:
# load precursor and protein data from diann with alphapepttools functions
prec_adata = at.io.read_psm_table(prec_report_path, level="psm", search_engine="diann", 
                                    intensity_column="Precursor.Quantity", feature_id_column="Precursor.Id",
                                    sample_id_column="Run", var_columns=["Protein.Group", "Protein.Ids"])

prot_adata = at.io.read_psm_table(prec_report_path, level="protein", search_engine="diann",
                                    intensity_column="PG.MaxLFQ", feature_id_column="Protein.Group",
                                    sample_id_column="Run"
)

In [77]:
# # load h5ad data saved by alphapepttools
# prec_adata = ad.read_h5ad("../data/albrecht.precursors.h5ad")
# prot_adata = ad.read_h5ad("../data/albrecht.proteins.h5ad")

In [78]:
#create mudata object
msdata = md.MuData(
    # These are the raw data levels
    {
        "protein_level": prot_adata,
        #"peptide_level": ad.AnnData(...),
        "precursor_level": prec_adata,
    },
)

  self._update_attr("var", axis=0, join_common=join_common)

  self._update_attr("obs", axis=1, join_common=join_common)



In [79]:
prot_adata.var

""
Protein.Group
A6NIH7
O00410
O60779
O75822
P09417
P27361
P28482
P36578
P37108


In [80]:
prec_adata.var

,Protein.Group,Protein.Ids
Precursor.Id,,
(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVR3,O75822,O75822
(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3,O75822,O75822
(UniMod:1)AAAAAAAVGGQQPSQPELPAPGLALDK3,Q6ZT12,Q6ZT12
(UniMod:1)AAAAAAGAASGLPGPVAQGLK2,Q96P70,Q96P70
(UniMod:1)AAAAAAGAASGLPGPVAQGLK3,Q96P70,Q96P70
(UniMod:1)AAAAAAGAGPEMVR2,P28482,P28482
(UniMod:1)AAAAAAGEAR2,P09417,P09417
(UniMod:1)AAAAAEEGMEPR2,P51788,P51788
(UniMod:1)AAAAAEQQQFYLLLGNLLSPDNVVR3,O00410,O00410


## implement on functions

In [89]:
psm_table = pd.read_csv(prec_report_path, sep="\t")

In [82]:
test_map = get_mapping_from_anndata(prec_adata, ["Protein.Group"])

In [83]:
test_map

,index,Protein.Group
0,(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVR3,O75822
1,(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3,O75822
2,(UniMod:1)AAAAAAAVGGQQPSQPELPAPGLALDK3,Q6ZT12
3,(UniMod:1)AAAAAAGAASGLPGPVAQGLK2,Q96P70
4,(UniMod:1)AAAAAAGAASGLPGPVAQGLK3,Q96P70
5,(UniMod:1)AAAAAAGAGPEMVR2,P28482
6,(UniMod:1)AAAAAAGEAR2,P09417
7,(UniMod:1)AAAAAEEGMEPR2,P51788
8,(UniMod:1)AAAAAEQQQFYLLLGNLLSPDNVVR3,O00410
9,(UniMod:1)AAAAAETPEVLR2,Q9NXW9


In [92]:
def get_unique_mappings(psm_path: str, feature_level_names: list[str]) -> pd.DataFrame:
    """
    Get unique mappings from PSM table for specified feature levels.
    
    Parameters:
    - psm_table: DataFrame containing PSM data with columns for each feature level.
    - feature_level_names: List of column names corresponding to feature levels (e.g., ["Precursor", "Protein"]).
    
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    # Select only the relevant columns for mapping
    psm_df = pd.read_csv(psm_path, sep="\t")
    mapping_df = psm_df.loc[:, feature_level_names].drop_duplicates()
    
    return mapping_df

In [94]:
test_map = get_unique_mappings(prec_report_path, ["Precursor.Id", "Protein.Group"])

In [97]:
def sparse_matrix_mapping(unique_mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    mapping_df : pd.DataFrame
        DataFrame where the index contains source features (e.g., precursors)
        and each column contains target features (e.g., protein groups).
        Produced by get_unique_mappings().

    Returns
    -------
    csr_matrix
        Square adjacency matrix of shape (n_total, n_total) where
        n_total = n_source + n_target features.
    """
    all_values = pd.unique(unique_mapping_df.values.ravel())                                                                                   
    value_to_idx = {v: i for i, v in enumerate(all_values)}                                                                     
    n = len(all_values)                                                                                                         
                                                                                                                                
    rows, cols = [], []                                                                                                         
    for _, row in unique_mapping_df.iterrows():                                                                                                
        for v1, v2 in combinations(row.values, 2):                                                                                     
            i, j = value_to_idx[v1], value_to_idx[v2]                                                                           
            rows.extend([i, j])                                                                                                 
            cols.extend([j, i])                                                                                                 
                                                                                                                                
    adj = sparse.coo_matrix(                                                                                                    
        (np.ones(len(rows), dtype=np.float64), (rows, cols)),
        shape=(n, n),                                                                                                           
    ).tocsr()                                                                                                                     
    adj.data = np.ones_like(adj.data)  # collapse summed duplicates to 1 
    adj = pd.DataFrame.sparse.from_spmatrix(adj, index=all_values, columns=all_values)                                                                                                                      
    return adj


In [98]:
matrix_varp = sparse_matrix_mapping(test_map)

In [99]:
matrix_varp

,AAAAAAAAAAAAAAAASAGGKEAASGPNDS3,P0CG40,AAAAAAAAAPAAAATAPTTAATTAATAAQ3,P37108,(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVR3,O75822,(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3,AAAAAAALQAK2,P36578,(UniMod:1)AAAAAAAVGGQQPSQPELPAPGLALDK3,...,(UniMod:1)AAAAAEQQQFYLLLGNLLSPDNVVR3,O00410,(UniMod:1)AAAAAETPEVLR2,Q9NXW9,(UniMod:1)AAAAAMAEQESAR2,Q7L5D6,(UniMod:1)AAAAAQGGGGGEPR2,P27361,AAAAASAAGPGGLVAGK2,A6NIH7
AAAAAAAAAAAAAAAASAGGKEAASGPNDS3,0,1.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P0CG40,1.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAAAAAAAAPAAAATAPTTAATTAATAAQ3,0,0,0,1.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P37108,0,0,1.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVR3,0,0,0,0,0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
O75822,0,0,0,0,1.0,0,1.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
(UniMod:1)AAAAAAAGDSDSWDADAFSVEDPVRK3,0,0,0,0,0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAAAAAALQAK2,0,0,0,0,0,0,0,0,1.0,0,...,0,0,0,0,0,0,0,0,0,0
P36578,0,0,0,0,0,0,0,1.0,0,0,...,0,0,0,0,0,0,0,0,0,0
(UniMod:1)AAAAAAAVGGQQPSQPELPAPGLALDK3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [100]:
msdata.varp["precursor_to_protein"] = matrix_varp.reindex(index=msdata.var_names, columns=msdata.var_names)

In [73]:
msdata

MuData object with n_obs × n_vars = 6 × 30
  varp:	'precursor_to_protein'
  2 modalities
    protein_level:	6 x 14
    precursor_level:	6 x 16
      var:	'Protein.Group', 'Protein.Ids'

In [107]:
def create_mudata_diann(psm_path: str, feature_level_names: list[str]) -> md.MuData:
    """
    create  a MuData object from the PSM table, including unique mappings between feature levels.
   
   Parameters:
    - psm_path: Path to the Diann PSM table file (e.g., TSV or CSV).
    - feature_level_names: List of column names in the PSM table that represent the feature levels to map (e.g., ["Precursor.Id", "Protein.Group"]).
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    #Create mapping between feature levels
    mapping_df = get_unique_mappings(psm_path, feature_level_names)
    matrix_varp = sparse_matrix_mapping(mapping_df)

    # load precursor and protein data from diann with alphapepttools functions
    prec_adata = at.io.read_psm_table(psm_path, level="psm", search_engine="diann", 
                                    intensity_column="Precursor.Quantity", feature_id_column="Precursor.Id",
                                    sample_id_column="Run", var_columns=feature_level_names
                                    )

    prot_adata = at.io.read_psm_table(psm_path, level="protein", search_engine="diann",
                                    intensity_column="PG.MaxLFQ", feature_id_column="Protein.Group",
                                    sample_id_column="Run"
                                    )
    
    #create mudata object
    mudata = md.MuData(
    # These are the raw data levels
        {
            "protein_level": prot_adata,
            #"peptide_level": ad.AnnData(...),
            "precursor_level": prec_adata,
        },
    )

    mudata.varp["precursor_to_protein"] = matrix_varp.reindex(index=mudata.var_names, columns=mudata.var_names)

    return mudata

In [108]:
msdata_function = create_mudata_diann(prec_report_path, ["Precursor.Id", "Protein.Group"])

  self._update_attr("var", axis=0, join_common=join_common)

  self._update_attr("obs", axis=1, join_common=join_common)



In [109]:
msdata_function

MuData object with n_obs × n_vars = 6 × 30
  varp:	'precursor_to_protein'
  2 modalities
    protein_level:	6 x 14
    precursor_level:	6 x 16
      var:	'Protein.Group'

In [110]:
msdata

MuData object with n_obs × n_vars = 6 × 30
  varp:	'precursor_to_protein'
  2 modalities
    protein_level:	6 x 14
    precursor_level:	6 x 16
      var:	'Protein.Group', 'Protein.Ids'